In [ ]:
!pip install --upgrade datasets pandas numpy gradio faiss-cpu scikit-learn fastapi uvicorn requests nest_asyncio
!pip install --upgrade sentence-transformers keybert spacy transformers
!python -m spacy download en_core_web_sm

In [ ]:
import os
import time
import pandas as pd
import numpy as np
import faiss
import spacy
import uvicorn
import nest_asyncio
from threading import Thread
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from keybert import KeyBERT

# Bypass async locks in Colab
nest_asyncio.apply()

app = FastAPI(title="AI Research Paper Analytics Engine", version="2.0")

print("[INFO] Booting Backend Neural Engine and Loading 50k Papers...")
if not os.path.exists("../data"):
    os.makedirs("../data")

CSV_PATH = "../data/cleaned_arxiv_papers.csv"
INDEX_PATH = "../data/paper_faiss.index"
EMBEDDING_PATH = "../data/arxiv_embeddings.npy"

# Ultra Stable Data Loader
if os.path.exists(CSV_PATH):
    df = pd.read_csv(CSV_PATH)
else:
    raw_dataset = load_dataset("CShorten/ML-ArXiv-Papers", split='train')
    df = pd.DataFrame(raw_dataset)[['title', 'abstract']].head(50000)
    df['title'] = df['title'].fillna("").astype(str)
    df['abstract'] = df['abstract'].fillna("").astype(str)
    df['paper_text'] = df['title'] + " " + df['abstract']
    df['paper_text'] = df['paper_text'].str.replace("\n", " ", regex=False).str.strip()
    df.to_csv(CSV_PATH, index=False)

# Activating Core ML Objects
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
kw_model = KeyBERT()
nlp = spacy.load("en_core_web_sm")

# ValueError Guarded Encoding Block
if os.path.exists(INDEX_PATH) and os.path.exists(EMBEDDING_PATH):
    index = faiss.read_index(INDEX_PATH)
    embeddings = np.load(EMBEDDING_PATH)
else:
    print("[PROCESS] Transforming strings into vector space matrix...")
    text_corpus = [str(text) for text in df['paper_text'].tolist() if len(str(text).strip()) > 0]
    embeddings = model.encode(text_corpus, batch_size=64, show_progress_bar=True)
    embeddings = np.array(embeddings).astype('float32')

    faiss_vectors = embeddings.copy()
    faiss.normalize_L2(faiss_vectors)
    index = faiss.IndexFlatIP(384)
    index.add(faiss_vectors)
    faiss.write_index(index, INDEX_PATH)
    np.save(EMBEDDING_PATH, embeddings)

themes = ["Computer Vision", "Natural Language Processing", "Reinforcement Learning", "Generative AI"]
if 'assigned_theme' not in df.columns:
    theme_vectors = model.encode(themes)
    theme_vectors = np.array(theme_vectors).astype('float32')
    faiss.normalize_L2(theme_vectors)
    norm_embeddings = embeddings / np.linalg.norm(embeddings, axis=1, keepdims=True)
    similarity_matrix = np.dot(norm_embeddings, theme_vectors.T)
    df['assigned_theme'] = [themes[idx] for idx in np.argmax(similarity_matrix, axis=1)]
    df.to_csv(CSV_PATH, index=False)

def generate_bullet_summary(text):
    sentences = str(text).split('. ')
    bullets = [f"• {s.strip()}" for s in sentences if len(s.strip()) > 35]
    return "\n".join(bullets[:3]) if bullets else "• Abstract data content available inside textual layer."

def extract_named_entities(text):
    doc = nlp(str(text))
    entities = [f"`{ent.text} ({ent.label_})`" for ent in doc.ents if ent.label_ in ['ORG', 'PRODUCT', 'WORK_OF_ART', 'PERCENT', 'CARDINAL']]
    unique_ents = list(set(entities))[:4]
    return " | ".join(unique_ents) if unique_ents else "`No explicit scientific entities found`"

class QuerySchema(BaseModel):
    query: str
    threshold: float

@app.post("/api/search")
def search_papers(payload: QuerySchema):
    try:
        start_time = time.time()
        clean_query = payload.query.strip().lower()
        query_vector = model.encode([clean_query]).astype('float32')
        faiss.normalize_L2(query_vector)
        scores, indices = index.search(query_vector, 15)
        latency = (time.time() - start_time) * 1000

        results = []
        valid_count = 0
        for rank in range(15):
            idx = int(indices[0][rank])
            score = float(scores[0][rank]) * 100
            if score < payload.threshold:
                continue
            valid_count += 1
            target_text = df['paper_text'].iloc[idx]
            abstract_text = df['abstract'].iloc[idx]

            extracted_tags = kw_model.extract_keywords(target_text, keyphrase_ngram_range=(1, 2), stop_words='english', top_n=3)
            formatted_tags = " | ".join([f"`{tag[0]}`" for tag in extracted_tags])

            bullet_summary = generate_bullet_summary(abstract_text)
            ner_entities = extract_named_entities(abstract_text)

            results.append({
                "rank": valid_count,
                "confidence": f"{score:.2f}%",
                "title": df['title'].iloc[idx],
                "theme": df['assigned_theme'].iloc[idx],
                "keywords": formatted_tags,
                "entities": ner_entities,
                "summary": bullet_summary,
                "abstract": abstract_text[:250] + "..."
            })
            if valid_count == 5:
                break
        return {"status": "success", "latency_ms": f"{latency:.2f}", "records_scanned": len(df), "matches_found": valid_count, "data": results}
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

@app.get("/api/theme/{selected_theme}")
def get_theme_papers(selected_theme: str):
    subset = df[df['assigned_theme'] == selected_theme].head(5)
    records = []
    for row in subset.itertuples():
        records.append({"title": row.title, "summary": generate_bullet_summary(row.abstract)})
    return {"status": "success", "data": records}

def run_server():
    uvicorn.run(app, host="127.0.0.1", port=8000, log_level="warning")

server_thread = Thread(target=run_server)
server_thread.start()
print("[SUCCESS] FastAPI Backend is now perfectly running in the background on port 8000!")

In [ ]:
import gradio as gr
import requests

BACKEND_URL = "http://127.0.0.1:8000"

def client_faiss_search(query, threshold):
    if not query or not query.strip():
        return "⚠️ **Alert:** Prompt console cannot be blank.", "N/A"
    try:
        payload = {"query": query, "threshold": float(threshold)}
        response = requests.post(f"{BACKEND_URL}/api/search", json=payload)
        if response.status_code != 200:
            return f"❌ **Backend Fault:** {response.json().get('detail')}", "Error"
        res_data = response.json()
        markdown_stream = f"## 🔍 FAISS Vector Engine Results for: *'{query}'*\nFiltered with active threshold score constraint: `>= {threshold}%`\n\n---\n\n"
        for record in res_data['data']:
            markdown_stream += f"### 📄 [RESULT {record['rank']}] — Confidence: {record['confidence']}\n**📌 Title:** {record['title']}\n**📁 Vertical:** `{record['theme']}`\n**💡 KeyBERT Concepts:** {record['keywords']}\n**🧠 SpaCy NER Entities:** {record['entities']}\n**📝 Summary Insights:**\n{record['summary']}\n\n**📄 Snippet:** {record['abstract']}\n\n---\n\n"
        if not res_data['data']:
            markdown_stream += "ℹ️ **No matches located.** Please adjust your Relevancy Threshold slider lower."
        telemetry_log = f"⚡ Latency: {res_data['latency_ms']} ms | Total Corpus: {res_data['records_scanned']} entries | Matches: {res_data['matches_found']}"
        return markdown_stream, telemetry_log
    except Exception as e:
        return f"❌ **Connection Refused:** Error: {str(e)}", "Network Offline"

def client_theme_exploration(theme):
    try:
        response = requests.get(f"{BACKEND_URL}/api/theme/{theme}")
        if response.status_code != 200:
            return "❌ **Error loading vertical data.**"
        res_data = response.json()
        markdown_stream = f"## 📁 Active Vertical Domain: **{theme}**\n"
        for rank, item in enumerate(res_data['data'], 1):
            markdown_stream += f"### {rank}. {item['title']}\n**🧠 Summary:**\n{item['summary']}\n\n"
        return markdown_stream
    except Exception as e:
        return f"❌ **Network Error:** {str(e)}"

print("[INFO] Initializing Gradio Client Interface...")
themes_choices = ["Computer Vision", "Natural Language Processing", "Reinforcement Learning", "Generative AI"]

with gr.Blocks(theme="glass") as enterprise_portal:
    gr.Markdown("# 🤖 Enterprise Distributed AI Intelligence Hub")
    with gr.Tabs():
        with gr.TabItem("🔍 Intelligent FAISS Vector Search"):
            with gr.Row():
                input_box = gr.Textbox(lines=2, placeholder="Type your search here (e.g., neural network)...", label="Search Console Input Box", scale=4)
                slider_bar = gr.Slider(minimum=30, maximum=90, value=50, step=5, label="Confidence Limit (%)", scale=1)
                trigger_button = gr.Button("Query Distributed Web API Backend", variant="primary")
            output_markdown_block = gr.Markdown(label="Web Client System Response Stream")
            metrics_box = gr.Textbox(label=" Live API Endpoint Performance Telemetry Log Dashboard", interactive=False)
            trigger_button.click(fn=client_faiss_search, inputs=[input_box, slider_bar], outputs=[output_markdown_block, metrics_box])
        with gr.TabItem("📁 Automated Theme Cluster Panels"):
            dropdown_menu = gr.Dropdown(choices=themes_choices, value=themes_choices[0], label="Target Segment Vertical")
            trigger_filter_button = gr.Button("Fetch Segment Stream")
            filter_markdown_block = gr.Markdown(label="Cluster Console Live Logs")
            trigger_filter_button.click(fn=client_theme_exploration, inputs=dropdown_menu, outputs=filter_markdown_block)

enterprise_portal.launch(share=True, debug=False, prevent_thread_lock=True)